In [1]:
!pip install unsloth

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.4/82.4 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 28.4/28.4 MB 70.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 50.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 42.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 70.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 86.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 71.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.9/216.9 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 90.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 230.5/230.5 kB 13.0 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninst

In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

from unsloth import FastLanguageModel

import subprocess
import polars as pl
import torch
from google.colab import userdata  # Pro Google Colab
from kaggle_secrets import UserSecretsClient  # Pro Kaggle
from huggingface_hub import login, HfApi, DatasetCard, DatasetCardData
from datasets import load_dataset
from datasets import Dataset, DatasetDict
from transformers import AutoModelForCausalLM, AutoTokenizer
from tqdm import tqdm

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


### **1. LOAD SECRETS (Kaggle/Colab Fallback) & HF LOGIN**

In [3]:
try:
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HF_TOKEN")
    user_email = user_secrets.get_secret("user_email")
    user_name = user_secrets.get_secret("user_name")
except Exception:
    try:
        hf_token = userdata.get("HF_TOKEN")
        user_email = userdata.get("USER_EMAIL")
        user_name = userdata.get("USER_NAME")
    except Exception as e:
        print(f"Warning: Failed to load secrets automatically: {e}")

In [4]:
if hf_token:
    login(token=hf_token)
    print("Successfully logged into Hugging Face Hub!")
else:
    raise ValueError("HF_TOKEN was not found in Secrets!")

Successfully logged into Hugging Face Hub!


### **2. GIT CONFIGURATION***

In [5]:
def set_git_config(email, name):
    try:
        subprocess.run(
            ["git", "config", "--global", "user.email", email], check=True
        )
        subprocess.run(["git", "config", "--global", "user.name", name], check=True)
        print(f"Git configuration set for: {name} <{email}>")
    except subprocess.CalledProcessError as e:
        print(f"Error setting Git configuration: {e}")

In [6]:
if user_email and user_name:
    set_git_config(user_email, user_name)

Git configuration set for: Radim Közl <kozl@svmetal.cz>


### **3. SETUP REPO_ID AND INITIALIZATION**

In [7]:
api = HfApi()
repo_exists = False
hf_username = "KRadim"

In [8]:
DATASET_REPO_ID = f"{hf_username}/multilingual-text2cypher-2025"

### **4. LOADING AND MERGING THE ORIGINAL DATASET (TRAIN + TEST) INTO POLARS**

In [9]:
ds = load_dataset("neo4j/text2cypher-2025v1")
df_raw_train = pl.from_pandas(ds["train"].to_pandas())

README.md:   0%|          | 0.00/109 [00:00<?, ?B/s]

data/train.parquet: reconstructing file:   0%|          |  0.00B / 4.52MB            

data/train.parquet: downloading bytes:           |  0.00B            

data/test.parquet: reconstructing file:   0%|          |  0.00B /  594kB            

data/test.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/35946 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/4438 [00:00<?, ? examples/s]

In [10]:
if "test" in ds:
    df_raw_test = pl.from_pandas(ds["test"].to_pandas())
    df_raw = pl.concat([df_raw_train, df_raw_test])
    print(f"Merged! Train: {len(df_raw_train)}, Test: {len(df_raw_test)} | Total to process: {len(df_raw)}")
else:
    df_raw = df_raw_train
    print(f"There is no test split in the dataset; I am proceeding only with the train split ({len(df_raw)} rows).")

Merged! Train: 35946, Test: 4438 | Total to process: 40384


### **5. MODEL LOADING AND BATCH TRANSLATION**

In [11]:
max_seq_length = 2048  # Maximum context length
model_name = "unsloth/mistral-7b-instruct-v0.3-bnb-4bit"

In [12]:
print(f"\n--> [2/6] Loading model {model_name} onto GPU...")


--> [2/6] Loading model unsloth/mistral-7b-instruct-v0.3-bnb-4bit onto GPU...


In [13]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    load_in_4bit=True,
    dtype=None,
)

==((====))==  Unsloth 2026.10.3: Fast Mistral patching. Transformers: 5.16.1.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


/usr/local/lib/python3.13/dist-packages/huggingface_hub/constants.py:299: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/huggingface_hub/constants.py:299: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [14]:
# 2. Activating fast inference mode
FastLanguageModel.for_inference(model)

MistralForCausalLM(
  (model): MistralModel(
    (embed_tokens): Embedding(32768, 4096, padding_idx=770)
    (layers): ModuleList(
      (0-31): 32 x MistralDecoderLayer(
        (self_attn): MistralAttention(
          (q_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): MistralMLP(
          (gate_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear4bit(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): MistralRMSNorm((4096,), eps=1e-05)
        (post_attention_lay

In [15]:
BATCH_SIZE = 32

In [16]:
def translate_batch(questions: list[str], target_language: str) -> list[str]:
    prompts = [
        f"[INST] Translate the following text to {target_language}. Maintain technical terms. Direct translation only, no intro or markdown:\n{q} [/INST]"
        for q in questions
    ]

    inputs = tokenizer(
        prompts, 
        return_tensors="pt", 
        padding=True, 
        truncation=True,
        max_length=512
    ).to("cuda")

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=64,
            do_sample=False,
            use_cache=True,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    translations = []
    input_len = inputs["input_ids"].shape[1]
    
    for out in outputs:
        gen_tokens = out[input_len:]
        translated_text = tokenizer.decode(gen_tokens, skip_special_tokens=True).strip()
        translations.append(translated_text)

    return translations

#### --- ITERATIVE CHUNK PROCESSING (IN BATCHES OF 500 ROWS) ---

In [17]:
CHECKPOINT_PATH = "multilingual_checkpoint.parquet"
CHUNK_SIZE = 500

In [18]:
TARGET_LANGUAGES = {
    "question_cz": "Czech",
    "question_it": "Italian",
    "question_de": "German",
    "question_es": "Spanish",
    "question_fr": "French"
}

### **6. Attempt to pull the existing work-in-progress state directly from the target HF repository.**

In [19]:
try:
    print(f"🔄 Downloading the existing state of the dataset from the HF Hub ({DATASET_REPO_ID})...")
    ds_checkpoint = load_dataset(DATASET_REPO_ID, split="train")
    df_multilingual = pl.from_pandas(ds_checkpoint.to_pandas())
    print("✅ Work-in-progress state successfully loaded from HF!")
except Exception as e:
    print(f"🆕 Nothing exists on HF yet ({e}). I am creating a new structure from the original dataset...")
    df_multilingual = df_raw.rename({"question": "question_en"})
    for col_name in TARGET_LANGUAGES.keys():
        df_multilingual = df_multilingual.with_columns(
            pl.lit(None, dtype=pl.Utf8).alias(col_name)
        )

🔄 Downloading the existing state of the dataset from the HF Hub (KRadim/multilingual-text2cypher-2025)...


README.md:   0%|          | 0.00/773 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 5.22MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/40384 [00:00<?, ? examples/s]

✅ Work-in-progress state successfully loaded from HF!


In [20]:
total_rows = len(df_multilingual)
chunk_processed = False

In [21]:
print("\n--> Looking for an unfinished chunk (500 lines)...")


--> Looking for an unfinished chunk (500 lines)...


In [22]:
for chunk_start in range(0, total_rows, CHUNK_SIZE):
    chunk_end = min(chunk_start + CHUNK_SIZE, total_rows)
    
    # Check whether this chunk has already been translated into all languages
    chunk_needs_work = False
    for col_name in TARGET_LANGUAGES.keys():
        current_values = df_multilingual[col_name][chunk_start:chunk_end].to_list()
        if any(v is None for v in current_values):
            chunk_needs_work = True
            break
            
    if not chunk_needs_work:
        continue

    print(f"\n==================================================")
    print(f"🚀 Translating chunk: rows {chunk_start} to {chunk_end} of {total_rows}")
    print(f"==================================================")

    for col_name, lang_name in TARGET_LANGUAGES.items():
        chunk_questions = df_multilingual["question_en"][chunk_start:chunk_end].to_list()
        current_values = df_multilingual[col_name][chunk_start:chunk_end].to_list()
        
        if not any(v is None for v in current_values):
            print(f"⏩ Column '{col_name}' in this chunk is done, skipping...")
            continue

        print(f"Generating translation for {lang_name} ({col_name})...")
        translated_chunk = []
        
        for i in tqdm(range(0, len(chunk_questions), BATCH_SIZE)):
            batch = chunk_questions[i : i + BATCH_SIZE]
            translated_batch = translate_batch(batch, target_language=lang_name)
            translated_chunk.extend(translated_batch)

        # Safe writing to Polars
        updated_col = df_multilingual[col_name].to_list()
        updated_col[chunk_start:chunk_end] = translated_chunk
        
        df_multilingual = df_multilingual.with_columns(
            pl.Series(name=col_name, values=updated_col)
        )

    # 2. Saving the updated state to the same HF repository
    print(f"\n☁️ Uploading the updated state directly to the HF Hub ({DATASET_REPO_ID})...")
    ds_to_push = Dataset.from_polars(df_multilingual)
    ds_to_push.push_to_hub(
        repo_id=DATASET_REPO_ID,
        split="train",
        token=hf_token,
        private=False
    )
    print(f"✅ Chunk {chunk_start} to {chunk_end} successfully translated and saved to HF!")
    
    chunk_processed = True
    print("\n🛑 500 lines completed! Stopping the script for this run.")
    break  # We will stop after 1 chunk



🚀 Translating chunk: rows 500 to 1000 of 40384
Generating translation for Czech (question_cz)...


100%|██████████| 16/16 [07:01<00:00, 26.37s/it]


Generating translation for Italian (question_it)...


100%|██████████| 16/16 [06:59<00:00, 26.25s/it]


Generating translation for German (question_de)...


...

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

✅ Chunk 500 to 1000 successfully translated and saved to HF!

🛑 500 lines completed! Stopping the script for this run.


### Checking if anything remains to be translated

In [23]:
any_missing = False
for col_name in TARGET_LANGUAGES.keys():
    if df_multilingual[col_name].null_count() > 0:
        any_missing = True
        break

### **7. UPLOADING THE DATASET AND DATASET CARD (README) TO THE HF HUB**

In [24]:
if any_missing:
    print("\nℹ️ The script successfully completed a batch of 500 lines and stopped safely before the time limit.")
else:
    print("\n🎉 All chunks have been fully translated! I'm starting the final transformation and upload.")
    
    print("\n--> [4/6] Transforming data in Polars (Unpivot/Melt + Shuffle)...")
    df_flat = df_multilingual.unpivot(
        on=["question_en", "question_cz", "question_it", "question_de", "question_es", "question_fr"],
        index=["cypher", "schema", "data_source", "instance_id"],
        variable_name="language",
        value_name="question"
    )

    df_flat = df_flat.with_columns(
        pl.col("language").str.replace("question_", "")
    )

    df_final = df_flat.sample(fraction=1.0, shuffle=True, seed=42)

    print("\n--> Splitting into Train (90%) and Test (10%)...")
    test_size = int(len(df_final) * 0.10)
    df_test = df_final.head(test_size)
    df_train = df_final.tail(len(df_final) - test_size)

    print(f"Total rows: {len(df_final)} | Train: {len(df_train)} | Test: {len(df_test)}")

    print(f"\n--> [6/6] Uploading final dataset to repository: {DATASET_REPO_ID}...")
    dataset_dict = DatasetDict({
        "train": Dataset.from_polars(df_train),
        "test": Dataset.from_polars(df_test)
    })

    dataset_dict.push_to_hub(
        repo_id=DATASET_REPO_ID,
        token=hf_token,
        private=False
    )

    print("\n--> Creating and uploading Dataset Card (README.md)...")
    card_metadata = DatasetCardData(
        language=["en", "cs", "it", "de", "es", "fr"],
        license="apache-2.0",
        tags=["text-to-cypher", "neo4j", "multilingual", "synthetic", "mistral-7b-instruct-v0.3", "graph-db"],
        dataset_info={
            "features": [
                {"name": "language", "dtype": "string"},
                {"name": "question", "dtype": "string"},
                {"name": "cypher", "dtype": "string"},
                {"name": "schema", "dtype": "string"},
                {"name": "data_source", "dtype": "string"},
                {"name": "instance_id", "dtype": "string"},
            ],
            "splits": [
                {"name": "train", "num_bytes": dataset_dict["train"].size_in_bytes, "num_examples": len(dataset_dict["train"])},
                {"name": "test", "num_bytes": dataset_dict["test"].size_in_bytes, "num_examples": len(dataset_dict["test"])},
            ],
        }
    )

    dataset_card_text = f"""
# Multilingual Text-to-Cypher Dataset (2025)

This dataset is an expanded, **multilingual synthetic edition** created by combining all original splits of [`neo4j/text2cypher-2025v1`](https://huggingface.co/datasets/neo4j/text2cypher-2025v1).

It is designed for fine-tuning Large Language Models (LLMs) to generate accurate **Neo4j Cypher queries** from natural language prompts across 6 languages:
- 🇬🇧 **English (`en`)**
- 🇨🇿 **Czech (`cz`)**
- 🇮🇹 **Italian (`it`)**
- 🇩🇪 **German (`de`)**
- 🇪🇸 **Spanish (`es`)**
- 🇫🇷 **French (`fr`)**

---

## Dataset Splits

- **`train`**: {len(dataset_dict['train'])} examples (90%)
- **`test`**: {len(dataset_dict['test'])} examples (10%)

---

## Creation Process

1. **Data Concatenation**: Combined all split data from `neo4j/text2cypher-2025v1`.
2. **Translation & Synthesis**: Translated queries into CZ, IT, DE, ES, and FR using **`mistral-7b-instruct-v0.3`**.
3. **Data Transformation**: Unpivoted using **Polars** (expanding row count 6x).
4. **Reshuffling & Splitting**: Globally shuffled with seed `42` and split into new `train` (90%) and `test` (10%) splits.
"""

    card = DatasetCard(dataset_card_text)
    card.data = card_metadata
    card.push_to_hub(repo_id=DATASET_REPO_ID, token=hf_token)

    print("\n" + "=" * 60)
    print(f"✅ Everything successfully completed!")
    print(f"The dataset is available at: https://huggingface.co/datasets/{DATASET_REPO_ID}")
    print("=" * 60)


ℹ️ The script successfully completed a batch of 500 lines and stopped safely before the time limit.
